# Feature Engineering Analysis

**Purpose:** Investigate potential new features that could improve model performance, based on insights from descriptive and inferential analysis.

**Tasks:** 

1. Create new features by combining existing ones (e.g., date-based features like "day of the week").
2. Encode categorical variables: Label encoding, One-Hot encoding, or similar techniques.
3. Feature scaling: Min-Max scaling, Standardization.
4. Feature selection techniques (e.g., based on correlation matrix, mutual information, feature importance).

In [1]:
import json

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.api as sm
import statsmodels.stats.api as sms

pd.set_option('display.max_columns', None)

In [2]:
# reading data
df = pd.read_csv(r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\data\raw\fraud test.csv")
df.shape

(555719, 23)

In [3]:
data = df[df.columns[1:]].copy()
data.head(3)

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,21/06/2020 12:14,2.291160e+15,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,SC,29209,33.9659,-80.9355,333497,Mechanical engineer,19/03/1968,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,21/06/2020 12:14,3.573030e+15,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,UT,84002,40.3207,-110.4360,302,"Sales professional, IT",17/01/1990,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,21/06/2020 12:14,3.598220e+15,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,Bellmore,NY,11710,40.6729,-73.5365,34496,"Librarian, public",21/10/1970,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0


In [4]:
cols_div_path = r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\notebooks\metadata\Credit_cols_classified.json"
with open(cols_div_path,'r') as file:
    cols_division = json.load(file)
cols_division

{'target_col': 'is_fraud',
 'uniq_cols': ['cc_num', 'trans_num'],
 'num_cols': ['amt', 'city_pop'],
 'cat_cols': ['merchant',
  'category',
  'first',
  'last',
  'gender',
  'street',
  'city',
  'state',
  'job'],
 'loc_cols': ['long', 'lat', 'merch_lat', 'merch_long', 'zip'],
 'time_cols': ['trans_date_trans_time', 'dob', 'unix_time']}

In [5]:
data["amt_per_pop"] = data["amt"] / (data["city_pop"] + 1e-6)
data["log_amt_per_pop"] = np.log1p(data["amt"]) - np.log1p(data["city_pop"])

The feature `log_amt_per_pop` helps capture:

- High transaction amount in small population → suspicious
- Same amount in large city → less suspicious

In [6]:
data.groupby(by = "is_fraud")["log_amt_per_pop"].describe().T

is_fraud,0,1
count,553574.000000,2145.000000
mean,-4.831561,-2.826417
std,2.783637,2.658438
min,-14.184395,-12.233963
25%,-6.640328,-4.592013
50%,-4.432446,-2.374136
75%,-2.776941,-0.800687
max,4.122553,3.559684


`first`, `last` and `street` can be dropped as it will not give any kind of usefull information.

In [18]:
from sklearn.preprocessing import OneHotEncoder,LabelEncoder

In [8]:
cat_cols_encoding = {
    'merchant': "dummie",
    'category': "dummie",
    'gender': "one_hot",
    'city': "label",
    'state': "label",
    'job': "label"
}

In [40]:
def fit_encode_data(data, col_encod_map, drop_first=True):
    data = data.copy()
    encoder_dict = {}

    for col, encod in col_encod_map.items():
        if col not in data.columns:
            continue

        if encod == "label":
            le = LabelEncoder()
            data[col] = le.fit_transform(data[col])

            encoder_dict[col] = {
                "type": "label",
                "encoder": le
            }

        elif encod == "one_hot":
            ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
            transformed = ohe.fit_transform(data[[col]])

            cols = [f"{col}_{c}" for c in ohe.categories_[0]]
            if drop_first:
                transformed = transformed[:, 1:]
                cols = cols[1:]

            df_ohe = pd.DataFrame(transformed, columns=cols, index=data.index)
            data = pd.concat([data.drop(columns=[col]), df_ohe], axis=1)

            encoder_dict[col] = {
                "type": "one_hot",
                "encoder": ohe,
                "columns": cols,
                "drop_first": drop_first
            }

        elif encod == "dummie":
            dummies = pd.get_dummies(data[col], prefix=col, drop_first=drop_first,dtype = int)
            data = pd.concat([data.drop(columns=[col]), dummies], axis=1)

            encoder_dict[col] = {
                "type": "dummie",
                "columns": dummies.columns
            }

    return data, encoder_dict

In [ ]:
# def fit_encode_data(data,col_encod_map:dict,drop_first=True):
#     encoder_dict = {}
#     for col, encod in col_encod_map.items():
#         if encod == "dummie":
#             dummie_encod = pd.get_dummies
#             dummie_encod(data, columns=[col],drop_first=drop_first)
#             encoder_dict[col] = dummie_encod
#         elif encod == "one_hot":
#             one_hot_encod = OneHotEncoder()
#             one_hot_encod.fit(data[[col]])
#             encoder_dict[col] = one_hot_encod
#         elif encod == "label":
#             label_encod = LabelEncoder()
#             label_encod.fit(data[[col]])
#             encoder_dict[col] = label_encod
#         else:
#             print("Encoder Not Implemented please try from the folowing:\n['dummie','one_hot','label']")
#     return encoder_dict


In [41]:
data,encod_dict = fit_encode_data(data,cat_cols_encoding)

In [42]:
data.head()

,trans_date_trans_time,cc_num,amt,first,last,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,amt_per_pop,log_amt_per_pop,merchant_fraud_Abbott-Steuber,merchant_fraud_Abernathy and Sons,merchant_fraud_Abshire PLC,"merchant_fraud_Adams, Kovacek and Kuhlman",merchant_fraud_Adams-Barrows,"merchant_fraud_Altenwerth, Cartwright and Koss",merchant_fraud_Altenwerth-Kilback,merchant_fraud_Ankunding LLC,merchant_fraud_Ankunding-Carroll,"merchant_fraud_Armstrong, Walter and Gottlieb",merchant_fraud_Auer LLC,merchant_fraud_Auer-Mosciski,merchant_fraud_Auer-West,merchant_fraud_Bahringer Group,"merchant_fraud_Bahringer, Bergnaum and Quitzon","merchant_fraud_Bahringer, Osinski and Block","merchant_fraud_Bahringer, Schoen and Corkery",merchant_fraud_Bahringer-Larson,merchant_fraud_Bahringer-Streich,merchant_fraud_Bailey-Morar,merchant_fraud_Balistreri-Nader,merchant_fraud_Barrows PLC,merchant_fraud_Bartoletti and Sons,merchant_fraud_Bartoletti-Wunsch,merchant_fraud_Barton Inc,merchant_fraud_Barton LLC,merchant_fraud_Bashirian Group,merchant_fraud_Bauch-Blanda,merchant_fraud_Bauch-Raynor,merchant_fraud_Baumbach Ltd,"merchant_fraud_Baumbach, Feeney and Morar","merchant_fraud_Baumbach, Hodkiewicz and Walsh","merchant_fraud_Baumbach, Strosin and Nicolas",merchant_fraud_Bechtelar-Rippin,"merchant_fraud_Becker, Harris and Harvey",merchant_fraud_Bednar Group,merchant_fraud_Bednar Inc,merchant_fraud_Bednar PLC,merchant_fraud_Beer-Jast,merchant_fraud_Beier LLC,merchant_fraud_Beier and Sons,merchant_fraud_Beier-Hyatt,merchant_fraud_Berge LLC,"merchant_fraud_Berge, Kautzer and Harris",merchant_fraud_Berge-Hills,merchant_fraud_Berge-Ullrich,merchant_fraud_Bernhard Inc,"merchant_fraud_Bernhard, Grant and Langworth",merchant_fraud_Bernhard-Lesch,merchant_fraud_Bernier and Sons,"merchant_fraud_Bernier, Streich and Jewess","merchant_fraud_Bernier, Volkman and Hoeger","merchant_fraud_Bins, Balistreri and Beatty",merchant_fraud_Bins-Howell,merchant_fraud_Bins-Rice,merchant_fraud_Bins-Tillman,merchant_fraud_Block Group,merchant_fraud_Block-Hauck,merchant_fraud_Block-Parisian,merchant_fraud_Bode-Rempel,merchant_fraud_Bode-Schuster,"merchant_fraud_Boehm, Block and Jakubowski","merchant_fraud_Boehm, Predovic and Reinger",merchant_fraud_Bogisich Inc,merchant_fraud_Bogisich-Homenick,merchant_fraud_Bogisich-Weimann,merchant_fraud_Botsford Ltd,merchant_fraud_Botsford PLC,merchant_fraud_Botsford and Sons,merchant_fraud_Boyer PLC,merchant_fraud_Boyer-Haley,merchant_fraud_Boyer-Reichert,merchant_fraud_Bradtke PLC,"merchant_fraud_Bradtke, Torp and Bahringer",merchant_fraud_Breitenberg LLC,merchant_fraud_Breitenberg-Hermiston,merchant_fraud_Brekke and Sons,merchant_fraud_Brown Inc,merchant_fraud_Brown PLC,"merchant_fraud_Brown, Homenick and Lesch",merchant_fraud_Brown-Greenholt,merchant_fraud_Bruen-Yost,merchant_fraud_Buckridge PLC,merchant_fraud_Carroll PLC,merchant_fraud_Cartwright PLC,merchant_fraud_Cartwright-Harris,"merchant_fraud_Casper, Hand and Zulauf",merchant_fraud_Cassin-Harvey,merchant_fraud_Champlin and Sons,"merchant_fraud_Champlin, Rolfson and Connelly",merchant_fraud_Champlin-Casper,"merchant_fraud_Christiansen, Goyette and Schamberger",merchant_fraud_Christiansen-Gusikowski,merchant_fraud_Cole PLC,"merchant_fraud_Cole, Hills and Jewess",merchant_fraud_Collier Inc,merchant_fraud_Collier LLC,merchant_fraud_Connelly PLC,"merchant_fraud_Connelly, Reichert and Fritsch",merchant_fraud_Connelly-Carter,merchant_fraud_Conroy Ltd,"merchant_fraud_Conroy, Balistreri and Gorczany",merchant_fraud_Conroy-Cruickshank,merchant_fraud_Conroy-Emard,merchant_fraud_Cormier LLC,"merchant_fraud_Cormier, Stracke and Thiel",merchant_fraud_Corwin-Collins,merchant_fraud_Corwin-Gorczany,merchant_fraud_Corwin-Romaguera,"merchant_fraud_Cremin, Hamill and Reichel","merchant_fraud_Crist, Jakubowski and Littel",merchant_fraud_Crona and Sons,"merchant_fraud_Cronin, Kshlerin and Weber",merchant_fraud_Crooks and Sons,merchant_fraud_Cruickshank

In [44]:
encod_dict

{'city': {'type': 'label', 'encoder': LabelEncoder()},
 'state': {'type': 'label', 'encoder': LabelEncoder()},
 'job': {'type': 'label', 'encoder': LabelEncoder()}}

In [43]:
def encode_data(data, encoder_dict):
    data = data.copy()

    for col, info in encoder_dict.items():
        if info["type"] == "label":
            le = info["encoder"]
            data[col] = le.transform(data[col])

        elif info["type"] == "one_hot":
            ohe = info["encoder"]
            transformed = ohe.transform(data[[col]])

            cols = [f"{col}_{c}" for c in ohe.categories_[0]]

            if info["drop_first"]:
                transformed = transformed[:, 1:]
                cols = cols[1:]

            df_ohe = pd.DataFrame(transformed, columns=cols, index=data.index)
            data = pd.concat([data.drop(columns=[col]), df_ohe], axis=1)

        elif info["type"] == "dummie":
            dummies = pd.get_dummies(data[col], prefix=col)

            # Align with training columns
            for c in info["columns"]:
                if c not in dummies:
                    dummies[c] = 0

            dummies = dummies[info["columns"]]

            data = pd.concat([data.drop(columns=[col]), dummies], axis=1)

    return data

In [45]:
encode_data(data,encod_dict)

,trans_date_trans_time,cc_num,amt,first,last,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,amt_per_pop,log_amt_per_pop,merchant_fraud_Abbott-Steuber,merchant_fraud_Abernathy and Sons,merchant_fraud_Abshire PLC,"merchant_fraud_Adams, Kovacek and Kuhlman",merchant_fraud_Adams-Barrows,"merchant_fraud_Altenwerth, Cartwright and Koss",merchant_fraud_Altenwerth-Kilback,merchant_fraud_Ankunding LLC,merchant_fraud_Ankunding-Carroll,"merchant_fraud_Armstrong, Walter and Gottlieb",merchant_fraud_Auer LLC,merchant_fraud_Auer-Mosciski,merchant_fraud_Auer-West,merchant_fraud_Bahringer Group,"merchant_fraud_Bahringer, Bergnaum and Quitzon","merchant_fraud_Bahringer, Osinski and Block","merchant_fraud_Bahringer, Schoen and Corkery",merchant_fraud_Bahringer-Larson,merchant_fraud_Bahringer-Streich,merchant_fraud_Bailey-Morar,merchant_fraud_Balistreri-Nader,merchant_fraud_Barrows PLC,merchant_fraud_Bartoletti and Sons,merchant_fraud_Bartoletti-Wunsch,merchant_fraud_Barton Inc,merchant_fraud_Barton LLC,merchant_fraud_Bashirian Group,merchant_fraud_Bauch-Blanda,merchant_fraud_Bauch-Raynor,merchant_fraud_Baumbach Ltd,"merchant_fraud_Baumbach, Feeney and Morar","merchant_fraud_Baumbach, Hodkiewicz and Walsh","merchant_fraud_Baumbach, Strosin and Nicolas",merchant_fraud_Bechtelar-Rippin,"merchant_fraud_Becker, Harris and Harvey",merchant_fraud_Bednar Group,merchant_fraud_Bednar Inc,merchant_fraud_Bednar PLC,merchant_fraud_Beer-Jast,merchant_fraud_Beier LLC,merchant_fraud_Beier and Sons,merchant_fraud_Beier-Hyatt,merchant_fraud_Berge LLC,"merchant_fraud_Berge, Kautzer and Harris",merchant_fraud_Berge-Hills,merchant_fraud_Berge-Ullrich,merchant_fraud_Bernhard Inc,"merchant_fraud_Bernhard, Grant and Langworth",merchant_fraud_Bernhard-Lesch,merchant_fraud_Bernier and Sons,"merchant_fraud_Bernier, Streich and Jewess","merchant_fraud_Bernier, Volkman and Hoeger","merchant_fraud_Bins, Balistreri and Beatty",merchant_fraud_Bins-Howell,merchant_fraud_Bins-Rice,merchant_fraud_Bins-Tillman,merchant_fraud_Block Group,merchant_fraud_Block-Hauck,merchant_fraud_Block-Parisian,merchant_fraud_Bode-Rempel,merchant_fraud_Bode-Schuster,"merchant_fraud_Boehm, Block and Jakubowski","merchant_fraud_Boehm, Predovic and Reinger",merchant_fraud_Bogisich Inc,merchant_fraud_Bogisich-Homenick,merchant_fraud_Bogisich-Weimann,merchant_fraud_Botsford Ltd,merchant_fraud_Botsford PLC,merchant_fraud_Botsford and Sons,merchant_fraud_Boyer PLC,merchant_fraud_Boyer-Haley,merchant_fraud_Boyer-Reichert,merchant_fraud_Bradtke PLC,"merchant_fraud_Bradtke, Torp and Bahringer",merchant_fraud_Breitenberg LLC,merchant_fraud_Breitenberg-Hermiston,merchant_fraud_Brekke and Sons,merchant_fraud_Brown Inc,merchant_fraud_Brown PLC,"merchant_fraud_Brown, Homenick and Lesch",merchant_fraud_Brown-Greenholt,merchant_fraud_Bruen-Yost,merchant_fraud_Buckridge PLC,merchant_fraud_Carroll PLC,merchant_fraud_Cartwright PLC,merchant_fraud_Cartwright-Harris,"merchant_fraud_Casper, Hand and Zulauf",merchant_fraud_Cassin-Harvey,merchant_fraud_Champlin and Sons,"merchant_fraud_Champlin, Rolfson and Connelly",merchant_fraud_Champlin-Casper,"merchant_fraud_Christiansen, Goyette and Schamberger",merchant_fraud_Christiansen-Gusikowski,merchant_fraud_Cole PLC,"merchant_fraud_Cole, Hills and Jewess",merchant_fraud_Collier Inc,merchant_fraud_Collier LLC,merchant_fraud_Connelly PLC,"merchant_fraud_Connelly, Reichert and Fritsch",merchant_fraud_Connelly-Carter,merchant_fraud_Conroy Ltd,"merchant_fraud_Conroy, Balistreri and Gorczany",merchant_fraud_Conroy-Cruickshank,merchant_fraud_Conroy-Emard,merchant_fraud_Cormier LLC,"merchant_fraud_Cormier, Stracke and Thiel",merchant_fraud_Corwin-Collins,merchant_fraud_Corwin-Gorczany,merchant_fraud_Corwin-Romaguera,"merchant_fraud_Cremin, Hamill and Reichel","merchant_fraud_Crist, Jakubowski and Littel",merchant_fraud_Crona and Sons,"merchant_fraud_Cronin, Kshlerin and Weber",merchant_fraud_Crooks and Sons,merchant_fraud_Cruickshank